In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("test")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/23 21:14:57 WARN Utils: Your hostname, CrisBook.local, resolves to a loopback address: 127.0.0.1; using 192.168.13.159 instead (on interface en0)
26/04/23 21:14:57 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/23 21:14:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/23 21:14:59 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
import os
import requests

def download_file(url: str, output_path: str):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(output_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

In [3]:
KEV_CSV_URL = "https://www.cisa.gov/sites/default/files/csv/known_exploited_vulnerabilities.csv"
kev_raw_path = "../data/raw/kev/known_exploited_vulnerabilities.csv"

download_file(KEV_CSV_URL, kev_raw_path)

print("KEV downloaded successfully")

KEV downloaded successfully


In [4]:
kev_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("../data/raw/kev/known_exploited_vulnerabilities.csv")
)

kev_raw.printSchema()
kev_raw.show(5, truncate=False)

root
 |-- cveID: string (nullable = true)
 |-- vendorProject: string (nullable = true)
 |-- product: string (nullable = true)
 |-- vulnerabilityName: string (nullable = true)
 |-- dateAdded: date (nullable = true)
 |-- shortDescription: string (nullable = true)
 |-- requiredAction: string (nullable = true)
 |-- dueDate: string (nullable = true)
 |-- knownRansomwareCampaignUse: string (nullable = true)
 |-- notes: string (nullable = true)
 |-- cwes: string (nullable = true)

+--------------+-------------+-----------------------+------------------------------------------------------------------------------------------------------+----------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [5]:
from pyspark.sql.functions import col, upper, trim

kev_df = (
    kev_raw
    .withColumnRenamed("cveID", "cve_id")
    .withColumnRenamed("vendorProject", "vendor_project")
    .withColumnRenamed("vulnerabilityName", "vulnerability_name")
    .withColumnRenamed("dateAdded", "kev_date_added")
    .withColumnRenamed("shortDescription", "short_description")
    .withColumnRenamed("requiredAction", "required_action")
    .withColumnRenamed("knownRansomwareCampaignUse", "known_ransomware_campaign_use")
    .withColumn("cve_id", upper(trim(col("cve_id"))))
)

In [6]:
kev_df = kev_df.select(
    "cve_id",
    "vendor_project",
    "product",
    "vulnerability_name",
    "kev_date_added",
    "short_description",
    "required_action",
    "known_ransomware_campaign_use"
)

kev_df.show(5, truncate=False)

+--------------+--------------+-----------------------+------------------------------------------------------------------------------------------------------+--------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------

In [7]:
print("Total rows:", kev_df.count())
print("Distinct CVEs:", kev_df.select("cve_id").distinct().count())

Total rows: 1579
Distinct CVEs: 1579


In [ ]:
kev_df.write.mode("overwrite").parquet("../data/silver/kev")

26/04/24 01:08:51 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 926655 ms exceeds timeout 120000 ms
26/04/24 01:08:51 WARN SparkContext: Killing executors is not supported by current scheduler.
26/04/24 01:08:58 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$